<a href="https://colab.research.google.com/github/rashdiwsl/InternTrapAI/blob/feature-agent/modelTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 — Install libraries
!pip install transformers datasets torch scikit-learn -q

In [2]:
# CELL 2 — Import everything
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from torch.utils.data import Dataset

print("Libraries loaded!")
print("GPU available:", torch.cuda.is_available())

Libraries loaded!
GPU available: True


In [6]:
df = pd.read_csv(
    "final_dataset.csv",
    engine="python",
    on_bad_lines="skip"
)

print(df.head())
print(len(df))

                                                text  label
0  Congratulations! You have been selected for a ...      1
1  URGENT: Limited slots! Apply now for UK visa j...      1
2  Dear candidate, we are hiring for online typin...      1
3  Vacancy at Dubai company! Salary AED 8000/mont...      1
4  Work from home data entry job. Rs. 800 per pag...      1
12102


In [7]:
# CELL 3 (FIXED) — Load and split dataset safely
import pandas as pd
from sklearn.model_selection import train_test_split

# Safe CSV loading — handles quotes, commas, encoding issues
try:
    df = pd.read_csv(
        'final_dataset.csv',
        on_bad_lines='skip',      # skip broken rows instead of crashing
        quoting=3,                # QUOTE_NONE — ignore quote characters
        encoding='utf-8',
        engine='python'
    )
    print("Loaded with python engine")
except Exception as e:
    print(f"First attempt failed: {e}")
    # Fallback: read raw lines manually
    rows = []
    with open('final_dataset.csv', 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if i == 0 or not line:
                continue
            # Last character should be ,0 or ,1
            if line.endswith(',1'):
                text = line[:-2].strip().strip('"')
                rows.append({'text': text, 'label': 1})
            elif line.endswith(',0'):
                text = line[:-2].strip().strip('"')
                rows.append({'text': text, 'label': 0})
    df = pd.DataFrame(rows)
    print(f"Loaded manually: {len(df)} rows")

# Keep only needed columns
if 'text' not in df.columns or 'label' not in df.columns:
    # Some datasets use different column names — rename
    cols = df.columns.tolist()
    print("Columns found:", cols)
    df = df.rename(columns={cols[0]: 'text', cols[-1]: 'label'})

# Clean
df = df[['text', 'label']].copy()
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 10]          # remove very short rows
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
df = df[df['label'].isin([0, 1])]           # keep only valid labels
df = df.drop_duplicates(subset='text')
df = df.reset_index(drop=True)

print(f"\nTotal rows loaded: {len(df)}")
print(f"Scam   (1): {df['label'].sum()}")
print(f"Legit  (0): {(df['label']==0).sum()}")
print(f"Sample row: {df['text'].iloc[0][:80]}")

# Split 80% train / 20% test
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")

Loaded with python engine

Total rows loaded: 4556
Scam   (1): 512
Legit  (0): 4044
Sample row: Congratulations! You have been selected for a data entry job. Salary Rs. 50000/m

Train: 3644 | Test: 912


In [8]:
# CELL 4 — Tokenize the text
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(
    list(train_df['text']),
    truncation=True, padding=True, max_length=256
)
test_encodings = tokenizer(
    list(test_df['text']),
    truncation=True, padding=True, max_length=256
)
print("Tokenization complete!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenization complete!


In [9]:
# CELL 5 — Create PyTorch dataset class
class ScamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx])
                for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ScamDataset(
    train_encodings, list(train_df['label'])
)
test_dataset = ScamDataset(
    test_encodings, list(test_df['label'])
)
print("Datasets created!")

Datasets created!


In [12]:
# CELL 6 (FIXED) — Load model and train
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',        # ← FIXED (was evaluation_strategy)
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',             # ← stops wandb login prompt
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print("Starting training... (takes ~15-20 mins on GPU)")
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training... (takes ~15-20 mins on GPU)


Epoch,Training Loss,Validation Loss
1,0.079663,0.027589
2,0.063034,0.024717
3,0.009929,0.022971


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=684, training_loss=0.07743735049377408, metrics={'train_runtime': 300.0843, 'train_samples_per_second': 36.43, 'train_steps_per_second': 2.279, 'total_flos': 724066801053696.0, 'train_loss': 0.07743735049377408, 'epoch': 3.0})

In [13]:
# CELL 7 — Evaluate accuracy
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
print("ACCURACY:", accuracy_score(list(test_df['label']), preds))
print()
print(classification_report(
    list(test_df['label']), preds,
    target_names=['Legitimate','Scam']
))

ACCURACY: 0.993421052631579

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00       810
        Scam       0.97      0.97      0.97       102

    accuracy                           0.99       912
   macro avg       0.98      0.98      0.98       912
weighted avg       0.99      0.99      0.99       912



In [14]:
# CELL 8 — Save model and download it
model.save_pretrained('./scam_model')
tokenizer.save_pretrained('./scam_model')
print("Model saved!")

# Zip and download to your computer
import shutil
shutil.make_archive('scam_model', 'zip', './scam_model')
from google.colab import files
files.download('scam_model.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>